# 5 · The approval gate, and the background sweep

Two things that both come down to *when* control returns.

**The gate.** Blocking a card cannot be undone, so the run freezes before the
tool executes and waits for a person.

**The sweep.** 276 accounts one after another is a bottleneck, not a design.
Starting one returns a job id immediately.

Where the pieces go, because getting it backwards is the usual mistake:

- `HumanInTheLoopMiddleware` on the **disposition subagent**, where the dangerous tools are
- the checkpointer on the **supervisor**, because that is the run being frozen

In [ ]:
# Reload the package from disk on every run, so an edit to src/sentinel takes
# effect without restarting the kernel. Python caches imported modules in
# sys.modules and a stale one will happily report yesterday's numbers.
import sys, pathlib
for name in [m for m in sys.modules if m.startswith("sentinel")]:
    del sys.modules[name]

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("sentinel package:", ROOT / "src" / "sentinel")

## Starting a sweep returns before it does any work

One `SELECT DISTINCT`, one job row, one `Thread.start()`.

In [ ]:
import time
from sentinel import db
from sentinel.sweep import start_queue_sweep, check_sweep_status, collect_sweep_results, wait_for_sweep

db.init_runtime()

t0 = time.perf_counter()
job = start_queue_sweep(limit=3, workers=3)
elapsed = time.perf_counter() - t0

print(f"start_queue_sweep returned in {elapsed:.4f} s")
print("job id:", job)

In [ ]:
# The sweep is now running. check_sweep_status never blocks, so other questions
# are answerable while it works.
for _ in range(3):
    s = check_sweep_status(job)
    print(f"  {s['completed']}/{s['total']} done, {s['progress_pct']}%   status={s['status']}")
    time.sleep(5)

In [ ]:
results = wait_for_sweep(job, poll_seconds=10)
for d in results["dispositions"]:
    print(f"{d['account_id']}  {d['verdict']:<22} {d['confidence']}")

## The gate

`block_card` and `escalate_case` allow **approve** and **reject** only. No
`edit`: silently rewriting *which* card gets blocked is precisely the failure an
approval gate exists to prevent.

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import HumanInTheLoopMiddleware
from sentinel.tools.disposition_tools import DISPOSITION_TOOLS, IRREVERSIBLE
from sentinel.middleware import PolicyState, PolicyMiddleware
from sentinel.agents.disposition import PROMPT as DISPOSITION_PROMPT

saver = SqliteSaver(sqlite3.connect(":memory:", check_same_thread=False)); saver.setup()
officer = create_agent(
    init_chat_model("gpt-4.1-mini", model_provider="openai"),
    tools=DISPOSITION_TOOLS,
    system_prompt=DISPOSITION_PROMPT,
    middleware=[PolicyMiddleware(),
                HumanInTheLoopMiddleware(
                    interrupt_on={n: {"allowed_decisions": ["approve", "reject"]} for n in IRREVERSIBLE},
                    description_prefix="IRREVERSIBLE ACTION pending analyst approval")],
    state_schema=PolicyState, checkpointer=saver)

acct = results["dispositions"][0]["account_id"] if results["dispositions"] else "A00008"
cfg = {"configurable": {"thread_id": "gate-demo"}}
out = officer.invoke({"messages": [{"role": "user", "content":
    f"Account {acct} is an active takeover with money still moving on card K000080. "
    f"A verdict is already recorded. Block that card now."}],
    "account_id": acct, "unattended": False}, config=cfg)

print("interrupted:", bool(out.get("__interrupt__")))
for i in (out.get("__interrupt__") or []):
    print(getattr(i, "value", i))
print()
print("rows in the actions table while paused:",
      len(db.fetch("SELECT 1 FROM actions WHERE account_id = ?", (acct,))))

Nothing was written. No card is stopped. The whole run is frozen in the
checkpointer until somebody decides.

Both paths are generated as transcripts by `python -m sentinel.transcripts`.

In [ ]:
from langgraph.types import Command

resumed = officer.invoke(
    Command(resume={"decisions": [{"type": "reject",
        "message": "Refused by the analyst. Do not retry. Record the case without the action."}]}),
    config=cfg)
print(resumed["messages"][-1].text)
print()
print("action rows after the rejection:",
      len(db.fetch("SELECT 1 FROM actions WHERE account_id = ?", (acct,))))

During a sweep there is no human present, so irreversible actions are
*proposed and queued* rather than executed. An unattended run that could block
276 cards is a worse system than one that cannot.

In [ ]:
rows = db.fetch("SELECT * FROM actions WHERE status = 'proposed'")
for r in rows:
    print(f"#{r['action_id']}  {r['account_id']}  {r['action']} -> {r['target']}  [{r['status']}]")
print(f"{len(rows)} action(s) waiting for sign-off")